In [ ]:
## Load packages ----
import numpy as np
import pandas as pd
import sshtunnel
import psycopg2 as pg
import os
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'kirene',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'Kevindebruyne-2021!', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()


In [ ]:
## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

In [ ]:
## Request query ----
sql_query = f"""SELECT year, month, zip, county, res_ratio, bus_ratio, oth_ratio, tot_ratio
from NSAPH2.public.hud_zip2fips
where year = 2010 
limit 10;"""
test = pd.read_sql_query(sql_query, connection).reset_index()
test.head()

QC questions
* How many times (per year) are enrollment zipcodes not equal to admission zipcodes?
help misha compare zipcode in enrollments vs zipcode in admissions:: by beneficiary-year. Will compare beneficiary ID by year, compare their zipcodes on enrollments vs admission

* How many invalid codes are there in the diagnoses column?
- what approach should we follow to use the valid icd codes that misha shared invalid ICD codes - NSAPH Data Science Support Tasks

This effort leads to the creation of a master outcomes file like https://nsaph.github.io/handbook/analytic.html#:~:text=MedPar%20(-,Admissions,-)%23


## Count the number of times enrollment zip codes are not equal to admission zip codes per beneficiary-year

In [ ]:
## Request query ----
admission_difference = f"""SELECT
    e.year AS beneficiary_year,
    e.bene_id,
    e.zip AS enrollment_zip,
    a.zip AS admission_zip,
    COUNT(*) AS count_not_equal_zipcodes
FROM
    NSAPH2.medicare.enrollments AS e
JOIN
    NSAPH2.medicare.admissions AS a
ON
    e.bene_id = a.bene_id
    AND e.year = a.admission_year
WHERE
    e.year = 2008
    AND a.year = 2008
    AND e.zip <> a.zip
GROUP BY
    e.year, e.bene_id, e.zip, a.zip
ORDER BY
    e.year, e.bene_id
"""
admission_difference_df = pd.read_sql_query(admission_difference, connection).reset_index()
admission_difference_df.head()

In [ ]:
admission_difference_df.describe()

year, number of hospitalizations, number of hospitalization where zipcode is different

should not be very different across year 

In [ ]:
## Request query ----
admission_total = f"""SELECT count(distinct bene_id), year
    from NSAPH2.medicare.admissions 
GROUP BY
    year
"""
admission_total_df = pd.read_sql_query(admission_total, connection).reset_index()
admission_total_df.head()

## Website

https://www.cms.gov/medicare/coordination-benefits-recovery/overview/icd-code-lists

In [ ]:


-- Subquery 1: Split the ICD codes and unnest them
WITH icd_codes AS (
    SELECT
    unnest(string_to_array(regexp_replace(array_to_string(diagnoses, ','), '[\[\]"]', '', 'g'), ',')) AS extracted_icd_code
    FROM
        NSAPH2.medicare.admissions
    LIMIT 1000
)

select * 
from icd_codes
WHERE
    extracted_icd_code ~ '^[A-Z]\d{2,5}$'

